# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/imsahilahmed/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

**Lane:** Refresh / Content Opportunity Scoring (same lane as ML-02).

**Task type:** **Ranking / scoring** — not plain classification.

The decision is not "is this page declining?" alone. It is **which pages should an editor review first this week** when they can only look at ~20–50 pages. The output is a **ranked queue** with a priority score and action codes (refresh, expand, protect, prune, monitor). Classification shows up as a **proxy label** (`is_declining_label`) we can score against, but the product is an ordered list for human review — decision-support, not autopilot.

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

**What we score:** a **refresh priority score** — higher means "review sooner."

**Proxy label for evaluation:** `is_declining_label` = 1 when `trend_direction == "down"`, else 0.

**Where it comes from:** a **defined rule applied to observed measurements**. The inputs (`impressions_last_30d`, `impressions_prev_30d`) are measured in the data; the cut (`down` = more than a 20% drop) is a hand-written rule in the export pipeline. So this is a **proxy**, not a future outcome label. On the full warehouse later, I will rebuild the label from a forward-looking window — but on the starter CSV this is the honest starting point.

**Leakage guardrail:** because the label is built from `trend_direction` / `trend_pct`, those columns must **never** be model features — using them would just re-learn the rule.

## 3. Success metric

*One metric you can defend. What number means 'good'?*

**Metric:** **Precision@50** — of the top 50 pages in the ranked queue, what fraction truly have `is_declining_label == 1`?

**Why this metric:** it matches real editor capacity (roughly 20–50 reviews per week), punishes wasting scarce review time on false alarms, and can be computed today on a baseline without training a model first.

**What "good" means on the starter slice (measured, client-holdout from Week 1):**
- Hand-written baseline rule: **Precision@50 ≈ 0.24** (~12 of 50 correct)
- Random forest on the same signals: **Precision@50 ≈ 0.74** (~37 of 50 correct)

Beating the transparent baseline on this metric — on a proper grouped split — is the bar for "this lane is worth modeling." I will re-earn these numbers on the warehouse later; they are **directional** evidence from the 30k starter playground, not a final claim.

## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

**Unit of analysis:** one row = **one content page** (`content_id`) for one pseudonymized client (`client_id`), with trailing-90-day metrics aggregated on that page.

**Lane slice filter:** pages with `impressions_90d >= 100` — enough search visibility to rank meaningfully (same spirit as the pipeline's `measurable_opportunity` flag).

The code cell below loads the starter CSV, applies that slice, derives the proxy target, and shows what the modeling table looks like.

In [1]:
import os
import sys

import pandas as pd

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/imsahilahmed/flyrank-ml-internship"
REPO_DIR = "flyrank-ml-internship"

if IN_COLAB:
    import subprocess

    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("../..")
elif os.path.basename(os.getcwd()) == "work":
    os.chdir("..")

raw_path = "data/raw/content_refresh_anonymized.csv"
assert os.path.exists(raw_path), "starter CSV not found — are you at the repo root?"

df = pd.read_csv(raw_path)

# Refresh lane slice: enough GSC signal to prioritize
lane = df[df["impressions_90d"] >= 100].copy()

# Proxy target (same rule as scripts/01_prepare_features.py)
lane["is_declining_label"] = lane["trend_direction"].str.lower().eq("down").astype(int)

# Sketch of the modeling unit — features + proxy target
unit_cols = [
    "content_id",
    "client_id",
    "content_type",
    "impressions_90d",
    "clicks_90d",
    "ctr",
    "avg_position",
    "engagement_rate",
    "days_since_last_update",
    "is_declining_label",
]
unit = lane[unit_cols].copy()

print("Working dir:", os.getcwd())
print("Starter rows:", f"{len(df):,}")
print("Lane slice (impressions_90d >= 100):", f"{len(unit):,} rows")
print("Unit of analysis: one row = one content page")
print("Declining rate in slice:", f"{unit['is_declining_label'].mean():.1%}")
print()
print("Target column distribution:")
print(unit["is_declining_label"].value_counts().rename({0: "not declining (proxy)", 1: "declining (proxy)"}))
print()
unit.head(10)

Working dir: C:\flyrank-internship-ml\flyrank-ml-internship
Starter rows: 30,000
Lane slice (impressions_90d >= 100): 22,006 rows
Unit of analysis: one row = one content page
Declining rate in slice: 59.8%

Target column distribution:
is_declining_label
declining (proxy)        13152
not declining (proxy)     8854
Name: count, dtype: int64



,content_id,client_id,content_type,impressions_90d,clicks_90d,ctr,avg_position,engagement_rate,days_since_last_update,is_declining_label
0,content_304f48230142,client_f369cb89fc,keyword article,3803,29,0.76,10.6,5.88,20,1
1,content_a1fb4e703a9e,client_4e07408562,keyword article,15320,7,0.05,20.3,0.00,25,1
2,content_9aa793d4d895,client_7f2253d7e2,keyword article,12581,11,0.09,36.5,0.00,20,1
3,content_331d6c4de07b,client_19581e27de,keyword article,11751,58,0.49,6.2,1.28,22,0
4,content_d99b7a2d90ca,client_3fdba35f04,keyword article,19140,24,0.13,44.0,0.00,14,1
5,content_d4084a4bc775,client_f369cb89fc,keyword article,3970,1,0.03,8.5,0.00,20,1
7,content_a63219c6e95a,client_19581e27de,keyword article,1724,1,0.06,21.2,3.57,22,0
8,content_5e6c160719bc,client_6208ef0f77,keyword article,32574,29,0.09,46.0,5.88,20,1
9,content_c27558df2b0c,client_19581e27de,keyword article,1240,2,0.16,4.9,0.00,104,1
10,content_d8ee6cc6d642,client_19581e27de,keyword article,20919,324,1.55,2.2,6.75,104,0


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A single if-statement does not separate "needs review" from "fine." Week 1 EDA already showed:
- Word count barely differs between declining and growing pages.
- Search volume alone is a weak separator for real traffic.
- CTR varies by `content_type` even after controlling for volume — one threshold on CTR misses that structure.

The hand-written baseline combines a few signals with fixed weights and only gets **~24% Precision@50**. A random forest on the same (non-leaky) features gets **~74%** on a client-holdout split. The pattern is **real but tangled across many signals** — impressions, clicks, position, engagement, freshness, content type — and shifts by client. That is when ML earns its place: not to replace the editor, but to **rank a weekly review queue** so limited human time goes to pages that actually show decline signals first.

**Action tied to output:** an editor pulls the top of the queue and chooses refresh, expand, protect, prune, or monitor — the score decides **order**, not the final action.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.